In [0]:
from pyspark.sql.functions import col, avg, stddev, min, max, hour, dayofweek, month, year
import matplotlib.pyplot as plt
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampType
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count
import pandas as pd


# Chargement de dataset


In [0]:
from pyspark.sql.functions import col
from functools import reduce

all_files = dbutils.fs.ls("/Volumes/workspace/default/filestore/Data_as_paquets/")
numeric_cols = ["fare_amount", "trip_distance", "tip_amount", "total_amount",
                "passenger_count", "extra", "mta_tax", "tolls_amount",
                "improvement_surcharge", "congestion_surcharge", "airport_fee"]

df_list = []

for f in all_files:
    print("Reading:", f.name)
    df = spark.read.parquet(f.path)
    
    # cast numeric columns to double
    for c in numeric_cols:
        df = df.withColumn(c, col(c).cast("double"))
    
    df_list.append(df)

# union all files safely
df_pop = reduce(lambda d1,d2: d1.unionByName(d2, allowMissingColumns=True), df_list)

print("✅ All files combined safely")
print("Total rows:", df_pop.count())


# EDA

In [0]:
df_pop.count()

In [0]:
df_pop.printSchema()

Vérification des doublons 

In [0]:
# compter le nombre de lignes totales vs lignes distinctes
total_rows = df_pop.count()
distinct_rows = df_pop.distinct().count()

print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Nombre de doublons: {total_rows - distinct_rows}")


Statistiques descriptives générales

In [0]:
colonNum=["passenger_count","trip_distance","fare_amount","tip_amount","total_amount"]
df_pop.select(colonNum).describe().toPandas()


Vérification des valeurs manquantes

In [0]:
for c in df_pop.columns:
    print(c, df_pop.filter(col(c).isNull()).count())


Vérifier la cohérence des dates

In [0]:
from pyspark.sql.functions import col

df_invalid_dates = df_pop.filter(col("tpep_pickup_datetime") > col("tpep_dropoff_datetime"))
df_invalid_dates.select("tpep_pickup_datetime","tpep_dropoff_datetime").show()
df_invalid_dates.count()

# Nettoyage 

Supprimer les doublons

In [0]:
# drop complete duplicates
df_pop = df_pop.dropDuplicates()

# vérifier
total_rows = df_pop.count()
distinct_rows = df_pop.distinct().count()
print(f"Total rows after removing duplicates: {total_rows}")
print(f"Distinct rows after removing duplicates: {distinct_rows}")


Inverser pickup & dropoff si pickup > dropoff


In [0]:
from pyspark.sql.functions import when, col

df_pop = df_pop.withColumn(
    "tpep_pickup_datetime",
    when(col("tpep_pickup_datetime") > col("tpep_dropoff_datetime"), col("tpep_dropoff_datetime"))
    .otherwise(col("tpep_pickup_datetime"))
)

df_pop = df_pop.withColumn(
    "tpep_dropoff_datetime",
    when(col("tpep_pickup_datetime") > col("tpep_dropoff_datetime"), col("tpep_pickup_datetime"))
    .otherwise(col("tpep_dropoff_datetime"))
)

df_pop.filter(col("tpep_pickup_datetime") > col("tpep_dropoff_datetime")).count()


Gérer les valeurs manquantes

In [0]:
df_pop = df_pop.withColumn(
    "passenger_count",
    when(col("passenger_count").isNull(), 1).otherwise(col("passenger_count"))
)
# vérifier s'il reste des nulls
df_pop.filter(col("passenger_count").isNull()).count()

In [0]:
df_pop = df_pop.withColumn(
    "store_and_fwd_flag",
    when(col("store_and_fwd_flag").isNull(), "N").otherwise(col("store_and_fwd_flag"))
)

# Vérifier
df_pop.filter(col("store_and_fwd_flag").isNull()).count()

In [0]:
df_pop = df_pop.withColumn(
    "congestion_surcharge",
    when(col("congestion_surcharge").isNull(), 0).otherwise(col("congestion_surcharge"))
)
# vérifier s'il reste des nulls
df_pop.filter(col("congestion_surcharge").isNull()).count()

In [0]:
df_pop = df_pop.withColumn(
    "airport_fee",
    when(col("airport_fee").isNull(), 0).otherwise(col("airport_fee"))
)
# vérifier s'il reste des nulls
df_pop.filter(col("airport_fee").isNull()).count()

In [0]:
for c in df_pop.columns:
    print(c, df_pop.filter(col(c).isNull()).count())


In [0]:
from pyspark.sql.functions import col, when

df_pop = df_pop.withColumn(
    "RatecodeID",
    when(col("RatecodeID").isNull(), 0) 
    .when( col("RatecodeID" ) == 99 , 0) 
    .otherwise(col("RatecodeID"))
)


# Analyses comparatives

**Prix moyen d’une course (fare_amount) :** Comparer l’estimation du sample vs la valeur exacte sur population



In [0]:
from pyspark.sql.functions import avg

df_pop.select(avg("fare_amount")).show()

**Distance moyenne d’une course (trip_distance) :** Identifier si l’échantillon est représentatif

In [0]:
df_pop.select(avg("trip_distance")).show()

**Durée moyenne des courses :** Calculer avec pickup/dropoff datetime

In [0]:
from pyspark.sql.functions import unix_timestamp

df = df.withColumn(
    "trip_duration_min",
    (unix_timestamp("tpep_dropoff_datetime") -
     unix_timestamp("tpep_pickup_datetime")) / 60
)

df.select(avg("trip_duration_min")).show()

**Proportion des courses avec tip > 0 :** Inférence vs valeur réelle

In [0]:
from pyspark.sql.functions import col

total_trips = df.count()
tip_trips = df.filter(col("tip_amount") > 0).count()

prop_tip_pop =tip_trips / total_trips
print(f"{prop_tip_pop * 100:.2f}%")


**Distribution des courses par heure/jour/semaine** Identifier les heures de pointe

In [0]:
#Tableau par heure
from pyspark.sql.functions import hour

df_hour = df_pop.groupBy(
    hour("tpep_pickup_datetime").alias("hour")
).count().orderBy("hour")
df_hour.show(25)


In [0]:
from pyspark.sql.functions import dayofweek, when

df_par_jour = df_pop.groupBy(
    dayofweek("tpep_pickup_datetime").alias("jour_semaine")
).count().orderBy("jour_semaine")

df_par_jour = df_par_jour.withColumn(
    "jour_nom",
    when(df_par_jour.jour_semaine == 1, "Dimanche")
    .when(df_par_jour.jour_semaine == 2, "Lundi")
    .when(df_par_jour.jour_semaine == 3, "Mardi")
    .when(df_par_jour.jour_semaine == 4, "Mercredi")
    .when(df_par_jour.jour_semaine == 5, "Jeudi")
    .when(df_par_jour.jour_semaine == 6, "Vendredi")
    .otherwise("Samedi")
)

df_par_jour.show()


In [0]:
#Distribution par semaine (numéro de semaine)
from pyspark.sql.functions import weekofyear

df_par_semaine = df_pop.groupBy(
    weekofyear("tpep_pickup_datetime").alias("semaine")
).count().orderBy("semaine")

df_par_semaine.show(1000)


**Comparaison des fares selon zones géographiques (pickup/dropoff boroughs) :** Identifier si l’échantillon reflète la diversité spatiale

In [0]:
from pyspark.sql.functions import avg,count

#Pickup LocationID (Population)
fare_by_PU = df_pop.groupBy("PULocationID") \
    .agg(
        avg("fare_amount").alias("avg_fare"),
        count("*").alias("nb_trips")
    ) \
    .orderBy("avg_fare", ascending=False)

fare_by_PU.show(10)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql.functions import col

fare_by_PU_filtered = fare_by_PU.filter(col("nb_trips") > 10000)
fare_pd = fare_by_PU_filtered.limit(10).toPandas()

plt.figure(figsize=(10,5))
sns.barplot(x="PULocationID", y="avg_fare", data=fare_pd)
plt.title("Prix moyen par zone de pickup (LocationID)")
plt.xlabel("Pickup LocationID")
plt.ylabel("Prix moyen ($)")
plt.show()


In [0]:
#Dropoff LocationID
avg_fare_DO = df_pop.groupBy("DOLocationID") \
    .agg(
        avg("fare_amount").alias("avg_fare"),
        count("*").alias("nb_trips")
    ) \
    .orderBy("avg_fare", ascending=False)

avg_fare_DO.show(10)


In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

top10_DO = avg_fare_DO.limit(10).toPandas()

plt.figure(figsize=(12,6))
sns.barplot(
    x="DOLocationID", 
    y="avg_fare", 
    data=top10_DO,
    palette="viridis"
)
plt.title("Top 10 Dropoff LocationID par prix moyen")
plt.xlabel("Dropoff LocationID")
plt.ylabel("Prix moyen ($)")
plt.show()

**Analyse des outliers :** Courses très longues ou très chères : impact sur estimation vs population

In [0]:
from pyspark.sql.functions import avg, col

# 1️⃣ Moyenne de trip_distance pour les courses >50 miles
long_trips_avg = df_pop.filter(col("trip_distance") > 50) \
    .agg(avg("trip_distance").alias("avg_long_distance")) \
    .collect()[0]["avg_long_distance"]

print("Distance moyenne des courses très longues (>50 miles):", long_trips_avg)

# 2️⃣ Moyenne de fare_amount pour les courses >200$
expensive_trips_avg = df_pop.filter(col("fare_amount") > 200) \
    .agg(avg("fare_amount").alias("avg_expensive_fare")) \
    .collect()[0]["avg_expensive_fare"]

print("Prix moyen des courses très chères (>200$):", expensive_trips_avg)


**Ratio tip/fare moyen par type de paiement (cash vs card)**

In [0]:
from pyspark.sql.functions import col, avg

pop_ratio = (
    df_pop
    .filter((col("payment_type") == 2) & (col("fare_amount") > 0))
    .withColumn("tip_ratio", col("tip_amount") / col("fare_amount"))
    .agg(avg("tip_ratio").alias("avg_tip_ratio"))
    .collect()[0]["avg_tip_ratio"]
)

print(f"Ratio moyen tip/fare (Cash): {pop_ratio:.9f}")


In [0]:
from pyspark.sql.functions import col, avg

pop_ratio = (
    df_pop
    .filter((col("payment_type") == 1) & (col("fare_amount") > 0))
    .withColumn("tip_ratio", col("tip_amount") / col("fare_amount"))
    .agg(avg("tip_ratio").alias("avg_tip_ratio"))
    .collect()[0]["avg_tip_ratio"]
)

print(f"Ratio moyen tip/fare (Card): {pop_ratio:.9f}")
